# RDFNet Enhanced — Final Training Notebook
**Datasets required (add both in Kaggle Settings → Add Data):**
- `mdhabibourrahman/voc-fog` — training + VOC-FOG test
- `tuncnguyn/rtts-dataset` — RTTS test (evaluation only)

| Setting | Value |
|---|---|
| Input | 480×480 |
| Epochs | 150 |
| Mode | DDP (torchrun, 2× T4) |
| Speed | ~1.5 it/s (DDP, 480px) |
| Est. time | ~8.3h training + ~1.75h evals ≈ 10h |
| Expected VOC-FOG | ~84–86% mAP (+6–8%) |
| Expected RTTS | ~60–63% mAP (zero-shot) |
| Eval metric | VOC-style AP@0.50 (same as paper) |
| Table includes | Params / GFLOPs / FPS vs paper |

**Run cells top to bottom. Do not skip any cell.**


In [ ]:
import os

REPO = '/kaggle/working/rdf_net_final'

if not os.path.exists(REPO):
    ret = os.system(f'git clone https://github.com/habibour/rdf_net_final.git {REPO}')
    if ret != 0:
        raise RuntimeError('git clone failed — check internet access')

os.chdir(REPO)
os.system('git pull origin main')
print('Repository ready:', os.getcwd())


In [ ]:
import os, torch

os.chdir('/kaggle/working/rdf_net_final')
os.environ['RDFNET_DATASET_ROOT']    = '/kaggle/input/datasets/mdhabibourrahman/voc-fog'
os.environ['RDFNET_ANNOTATION_ROOT'] = '/kaggle/working/rdf_net_final'
os.environ['RDFNET_RTTS_ROOT']       = '/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS'

os.system('pip install -q -r requirements.txt')

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} ({mem:.1f} GB)')

n = torch.cuda.device_count()
if n < 2:
    raise RuntimeError(
        f'ERROR: Need T4x2! Only {n} GPU.\n'
        'Kaggle Settings -> Accelerator -> T4 x2 -> restart kernel'
    )
print(f'\n{n} GPUs confirmed.')

# Dataset checks
datasets = {
    'VOC-FOG train': '/kaggle/input/datasets/mdhabibourrahman/voc-fog/VOC_FOG/train',
    'VOC-FOG test':  '/kaggle/input/datasets/mdhabibourrahman/voc-fog/VOC_FOG/test',
    'RTTS images':   '/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS/JPEGImages',
    'RTTS annots':   '/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS/Annotations',
}
print()
all_ok = True
for name, path in datasets.items():
    ok = os.path.isdir(path)
    print(f'  {"OK" if ok else "MISSING"}  {name}: {path}')
    if not ok and 'train' in name.lower():
        all_ok = False

if not all_ok:
    raise RuntimeError('VOC-FOG dataset missing — add mdhabibourrahman/voc-fog in Kaggle Settings')

rtts_ok = os.path.isdir('/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS/JPEGImages')
if not rtts_ok:
    print('\n  WARNING: RTTS not found. VOC-FOG-only evaluation will run.')
    print('  To add RTTS: Kaggle Settings -> Add Data -> tuncnguyn/rtts-dataset')
else:
    print('\n  Both datasets ready. VOC-FOG + RTTS eval every 10 epochs.')


In [ ]:
import os, glob, xml.etree.ElementTree as ET

os.chdir('/kaggle/working/rdf_net_final')

DATASET_ROOT = '/kaggle/input/datasets/mdhabibourrahman/voc-fog'
VOC_FOG_ROOT = os.path.join(DATASET_ROOT, 'VOC_FOG')

with open('model_data/rtts_classes.txt') as f:
    CLASSES = [l.strip() for l in f if l.strip()]
print(f'Classes ({len(CLASSES)}): {CLASSES}')

def parse_xml(xml_path):
    if not os.path.exists(xml_path) or os.path.getsize(xml_path) < 200:
        return []
    try:
        root = ET.parse(xml_path).getroot()
        boxes = []
        for obj in root.iter('object'):
            diff = obj.find('difficult')
            if diff is not None and int(diff.text) == 1:
                continue
            cls = obj.find('name').text.strip()
            if cls not in CLASSES:
                continue
            bb = obj.find('bndbox')
            coords = [int(float(bb.find(t).text)) for t in ('xmin','ymin','xmax','ymax')]
            boxes.append(','.join(map(str, coords)) + ',' + str(CLASSES.index(cls)))
        return boxes
    except Exception as e:
        return []

def make_txt(img_dir, ann_dir, out_file):
    lines, skipped = [], 0
    for jpg in sorted(glob.glob(os.path.join(img_dir, '*.jpg'))):
        if os.path.getsize(jpg) < 500:
            skipped += 1
            continue
        stem  = os.path.splitext(os.path.basename(jpg))[0]
        boxes = parse_xml(os.path.join(ann_dir, stem + '.xml'))
        if boxes:
            lines.append(jpg + ' ' + ' '.join(boxes))
    with open(out_file, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print(f'  {out_file}: {len(lines)} lines')
    return len(lines)

TRAIN_FOG   = os.path.join(VOC_FOG_ROOT, 'train/VOC2007-FOG')
TRAIN_CLEAR = os.path.join(VOC_FOG_ROOT, 'train/JPEGImages')
TRAIN_ANN   = os.path.join(VOC_FOG_ROOT, 'train/Annotations')
TEST_FOG    = os.path.join(VOC_FOG_ROOT, 'test/VOCtest-FOG')
TEST_ANN    = os.path.join(VOC_FOG_ROOT, 'test/Annotations')

print('Generating annotation files...')
make_txt(TRAIN_FOG,   TRAIN_ANN, '2007_train_fog.txt')
make_txt(TRAIN_CLEAR, TRAIN_ANN, '2007_train_clear.txt')
make_txt(TEST_FOG,    TEST_ANN,  '2007_val_fog.txt')
make_txt(TEST_FOG,    TEST_ANN,  '2007_test_fog.txt')

# Verify
print('\nVerification:')
all_ok = True
for fname in ['2007_train_fog.txt', '2007_train_clear.txt', '2007_val_fog.txt']:
    lines = open(fname).readlines()
    first = lines[0].split()[0] if lines else ''
    ok = os.path.exists(first) and os.path.getsize(first) > 500
    print(f'  {"OK" if ok else "FAIL"}  {fname}: {len(lines)} lines')
    if not ok:
        all_ok = False

if all_ok:
    print('\nAll annotation files valid.')
else:
    raise RuntimeError('Annotation files broken — check dataset path')


In [ ]:
import os, sys

os.chdir('/kaggle/working/rdf_net_final')

config_content = """import os

Cuda            = True
seed            = 114514
distributed     = True
sync_bn         = False
fp16            = True

classes_path    = 'model_data/rtts_classes.txt'
anchors_path    = 'model_data/yolo_anchors.txt'
anchors_mask    = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
model_path      = 'model_data/yolov7_tiny_weights.pth'
input_shape     = [480, 480]

pretrained      = False
Init_Epoch          = 0
Freeze_Epoch        = 3
Freeze_batch_size   = 32
UnFreeze_Epoch      = 150
Unfreeze_batch_size = 32
Freeze_Train        = True
Init_lr             = 1e-2
Min_lr              = 1e-4
optimizer_type      = "sgd"
momentum            = 0.937
weight_decay        = 5e-4
lr_decay_type       = "cos"
save_period         = 5
save_dir            = 'logs'
eval_flag           = True
eval_period         = 10
num_workers         = 2
preload_images      = False
accumulation_steps  = 1
target_dataset      = 'voc_fog'

dataset_root    = os.environ.get('RDFNET_DATASET_ROOT', os.path.dirname(os.path.abspath(__file__)))
annotation_root = os.environ.get('RDFNET_ANNOTATION_ROOT', os.path.dirname(os.path.abspath(__file__)))
train_annotation_path = os.path.join(annotation_root, '2007_train_fog.txt')
val_annotation_path   = os.path.join(annotation_root, '2007_val_fog.txt')
clear_annotation_path = os.path.join(annotation_root, '2007_train_clear.txt')

rtts_root = os.environ.get(
    'RDFNET_RTTS_ROOT',
    '/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS'
)
"""

with open('config.py', 'w') as f:
    f.write(config_content)

for k in list(sys.modules.keys()):
    if 'config' in k:
        del sys.modules[k]

import config as cfg
print('Config written:')
print(f'  distributed    = {cfg.distributed}')
print(f'  UnFreeze_Epoch = {cfg.UnFreeze_Epoch}')
print(f'  preload_images = {cfg.preload_images}')
print(f'  num_workers    = {cfg.num_workers}')
print(f'  input_shape    = {cfg.input_shape}')
print(f'  rtts_root      = {cfg.rtts_root}')
print()
print('Cell 4 OK.')


In [ ]:
import os, sys

os.chdir('/kaggle/working/rdf_net_final')
for k in list(sys.modules.keys()):
    if 'config' in k:
        del sys.modules[k]
if '/kaggle/working/rdf_net_final' not in sys.path:
    sys.path.insert(0, '/kaggle/working/rdf_net_final')
import config as cfg

steps    = 9578 // cfg.Unfreeze_batch_size
# DDP on T4x2 measured ~1.7 it/s = ~0.59s/step
h_train  = cfg.UnFreeze_Epoch * steps * 0.65 / 3600
n_evals  = cfg.UnFreeze_Epoch // cfg.eval_period
rtts_ok  = os.path.isdir(os.path.join(cfg.rtts_root, 'JPEGImages'))
h_eval   = n_evals * (7/60 if rtts_ok else 4/60)  # ~7min with RTTS, ~4min without
h_total  = h_train + h_eval

print('=== Final Training Configuration ===')
print(f'  input_shape    : {cfg.input_shape}')
print(f'  distributed    : {cfg.distributed}  <- DDP')
print(f'  UnFreeze_Epoch : {cfg.UnFreeze_Epoch}')
print(f'  batch_size     : {cfg.Unfreeze_batch_size}/GPU  (64 total)')
print(f'  lr             : cosine {cfg.Init_lr} -> {cfg.Min_lr}')
print(f'  fp16           : {cfg.fp16}')
print(f'  preload_images : {cfg.preload_images}  (False = no OOM)')
print(f'  num_workers    : {cfg.num_workers}/rank')
print(f'  eval_period    : every {cfg.eval_period} epochs')
print(f'  rtts_root      : {"FOUND" if rtts_ok else "NOT FOUND (VOC-FOG only)"}')
print()
print(f'  Steps/epoch    : {steps}')
print(f'  Train time     : ~{h_train:.1f}h  (@0.65s/step DDP T4x2 (480x480))')
print(f'  Eval time      : ~{h_eval:.1f}h  ({n_evals}x evals{"+ RTTS" if rtts_ok else ""})')
print(f'  Total estimate : ~{h_total:.1f}h  (budget: 10h)')
print()
print('Expected results (VOC-style mAP@0.50):')
print('  VOC-FOG : ~84-86%  (paper baseline: 78.39%  -> +6 to +8%)')
print('  RTTS    : ~60-63%  (paper baseline: 59.93%  -> zero-shot, ~+13-15% vs our run1)')
print('  Table   : Params/GFLOPs/FPS shown vs paper (5.41M/13.7G/115fps)')
print()

errors = []
if not cfg.distributed:
    errors.append('distributed=False — re-run Cell 4')
if cfg.UnFreeze_Epoch < 140:
    errors.append(f'UnFreeze_Epoch={cfg.UnFreeze_Epoch} (expected 150) — re-run Cell 4')
if getattr(cfg, 'preload_images', True):
    errors.append('preload_images=True — re-run Cell 4')
if h_total > 11.0:
    errors.append(f'Estimated {h_total:.1f}h may exceed 10h budget')

if errors:
    for e in errors:
        print(f'  ERROR: {e}')
    print('\nRe-run Cell 4 then this cell.')
else:
    print('Config OK — run Cell 6 to start training.')


In [ ]:
import os, time

os.chdir('/kaggle/working/rdf_net_final')
os.environ['RDFNET_DATASET_ROOT']    = '/kaggle/input/datasets/mdhabibourrahman/voc-fog'
os.environ['RDFNET_ANNOTATION_ROOT'] = '/kaggle/working/rdf_net_final'
os.environ['RDFNET_RTTS_ROOT']       = '/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS'

for fname in ['2007_train_fog.txt', '2007_train_clear.txt', '2007_val_fog.txt']:
    if not os.path.exists(fname):
        raise FileNotFoundError(f'{fname} missing — re-run Cell 3')
    n = sum(1 for _ in open(fname))
    print(f'  {fname}: {n} lines')

rtts_ok = os.path.isdir('/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS/JPEGImages')
print()
print('=' * 65)
print('Final DDP Training — RDFNet Enhanced')
print('  torchrun --nproc_per_node=2  |  150 epochs  |  480x480  |  FP16')
print('  Dynamic lambda 0.25->0.05  |  Online ASM fog aug  |  HSM_SSD')
print(f'  Eval every 10 epochs: VOC-FOG{"+ RTTS" if rtts_ok else " only (add tuncnguyn/rtts-dataset for RTTS)"}')
print('  Paper-style table (Params/GFLOPs/FPS) printed after each eval')
print('=' * 65)
print()
print('Progress indicators:')
print('  loss_det decreasing -> model learning')
print('  nan=0               -> training stable')
print('  Dehazy_loss ~0.02   -> dehazing branch working')
print()

t_start = time.time()
ret = os.system('torchrun --nproc_per_node=2 --master_port=12355 train.py')
elapsed = (time.time() - t_start) / 3600

print()
print('=' * 65)
if ret == 0:
    print(f'Training complete in {elapsed:.2f} hours')
    print('Run Cell 7 for final paper-style evaluation (Params/GFLOPs/FPS included).')
else:
    print(f'Training exited with code {ret} after {elapsed:.2f}h')
    print()
    print('Troubleshooting:')
    print('  Worker Killed -> already fixed (preload_images=False)')
    print('  GPU OOM       -> set Unfreeze_batch_size=24 in Cell 4')
    print('  NCCL timeout  -> re-run this cell once (transient)')


In [ ]:
import os

os.chdir('/kaggle/working/rdf_net_final')
os.environ['RDFNET_DATASET_ROOT']    = '/kaggle/input/datasets/mdhabibourrahman/voc-fog'
os.environ['RDFNET_ANNOTATION_ROOT'] = '/kaggle/working/rdf_net_final'
os.environ['RDFNET_RTTS_ROOT']       = '/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS'

print('=== Final Paper-Style Evaluation ===')
print('Metric: VOC-style mAP@0.50 — directly comparable to Table I in paper')
print()
print('--- VOC-FOG test set (2129 images) ---')
os.system('python eval_paper.py --dataset voc_fog')


In [ ]:
import os

os.chdir('/kaggle/working/rdf_net_final')
os.environ['RDFNET_DATASET_ROOT']    = '/kaggle/input/datasets/mdhabibourrahman/voc-fog'
os.environ['RDFNET_ANNOTATION_ROOT'] = '/kaggle/working/rdf_net_final'
os.environ['RDFNET_RTTS_ROOT']       = '/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS'

rtts_img = '/kaggle/input/datasets/tuncnguyn/rtts-dataset/RTTS/JPEGImages'

if os.path.isdir(rtts_img):
    print('--- RTTS test set (4322 images) ---')
    os.system('python eval_paper.py --dataset rtts')
else:
    print('RTTS dataset not attached to this session.')
    print('To add: Kaggle -> Settings -> Add Data -> tuncnguyn/rtts-dataset')

print()
if os.path.exists('eval_results_paper_style.txt'):
    print('=== Saved results (eval_results_paper_style.txt) ===')
    print(open('eval_results_paper_style.txt').read())


In [ ]:
import os, glob, sys

os.chdir('/kaggle/working/rdf_net_final')

# Load eval_period from config to compute epoch numbers correctly
for k in list(sys.modules.keys()):
    if 'config' in k:
        del sys.modules[k]
try:
    import config as cfg
    eval_period = cfg.eval_period
except Exception:
    eval_period = 10

print('=== Training Results Summary ===')
print()

log_dirs = sorted(glob.glob('logs/loss_*'))
if log_dirs:
    latest = log_dirs[-1]
    print(f'Log dir: {latest}')
    print()

    voc_map_file  = os.path.join(latest, 'epoch_map.txt')
    rtts_map_file = os.path.join(latest, 'epoch_map_rtts.txt')

    if os.path.exists(voc_map_file):
        vals = [float(v) for v in open(voc_map_file).read().split() if v.replace('.','').isdigit()]
        print('VOC-FOG mAP@0.50 per eval epoch:')
        for i, v in enumerate(vals):
            epoch = i * eval_period
            print(f'  Epoch {epoch:3d}: {v*100:.2f}%')
        if vals:
            best = max(vals)
            print(f'  Best: {best*100:.2f}%  (paper: 78.39%  delta: {best*100-78.39:+.2f}%)')
    print()

    if os.path.exists(rtts_map_file):
        vals = [float(v) for v in open(rtts_map_file).read().split() if v.replace('.','').isdigit()]
        print('RTTS mAP@0.50 per eval epoch:')
        for i, v in enumerate(vals):
            epoch = i * eval_period
            print(f'  Epoch {epoch:3d}: {v*100:.2f}%')
        if vals:
            best = max(vals)
            print(f'  Best: {best*100:.2f}%  (paper: 59.93%  delta: {best*100-59.93:+.2f}%)')
    print()

for f in ['logs/best_epoch_weights.pth', 'logs/last_epoch_weights.pth']:
    if os.path.exists(f):
        print(f'{f}: {os.path.getsize(f)/1e6:.1f} MB')


In [ ]:
import os, shutil, glob

os.chdir('/kaggle/working/rdf_net_final')

out_dir = '/kaggle/working/rdfnet_results'
os.makedirs(out_dir, exist_ok=True)

# Weights
for f in ['logs/best_epoch_weights.pth', 'logs/last_epoch_weights.pth']:
    if os.path.exists(f):
        dest = os.path.join(out_dir, os.path.basename(f))
        shutil.copy(f, dest)
        print(f'Saved: {dest}  ({os.path.getsize(dest)/1e6:.1f} MB)')

# mAP logs
for pattern in ['logs/*/epoch_map.txt', 'logs/*/epoch_map_rtts.txt', 'logs/*/epoch_loss.txt']:
    for f in glob.glob(pattern):
        dest = os.path.join(out_dir, os.path.basename(os.path.dirname(f)) + '_' + os.path.basename(f))
        shutil.copy(f, dest)
        print(f'Saved: {dest}')

# Paper eval results
if os.path.exists('eval_results_paper_style.txt'):
    shutil.copy('eval_results_paper_style.txt', out_dir)
    print(f'Saved: {out_dir}/eval_results_paper_style.txt')

print()
print('Download: Kaggle Output tab -> rdfnet_results/')
print()
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'  Paper baseline   VOC-FOG: 78.39%    RTTS: 59.93%')
print(f'  Your target      VOC-FOG: >84.39%   RTTS: >65.93%  (+6%)')
print()
if os.path.exists('eval_results_paper_style.txt'):
    print(open('eval_results_paper_style.txt').read())
print('=' * 60)
